In [2]:

# ═════════════════════════════════════════════════════════════════════════════
# CELL 2 — IMPORTS & GPU SETUP
# ═════════════════════════════════════════════════════════════════════════════

import os, gc, json, time, re, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

# ── Standard ML ─────────────────────────────────────────────────────────────
from sklearn.model_selection  import train_test_split
from sklearn.preprocessing    import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model     import LogisticRegression
from sklearn.svm              import LinearSVC
from sklearn.naive_bayes      import ComplementNB
from sklearn.pipeline         import Pipeline
from sklearn.metrics          import (accuracy_score, classification_report,
                                       confusion_matrix)

# ── Gradient Boosting ───────────────────────────────────────────────────────
import lightgbm as lgb

# ── Neural Embeddings ───────────────────────────────────────────────────────
from sentence_transformers import SentenceTransformer

# ── Deep Learning ───────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# ── HuggingFace ─────────────────────────────────────────────────────────────
from transformers import (AutoTokenizer,
                           AutoModelForSequenceClassification,
                           get_cosine_schedule_with_warmup)
from datasets import load_dataset

# ── GPU Check ───────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpus = torch.cuda.device_count()

print(f"✅ Device: {device}")
print(f"✅ GPUs available: {n_gpus}")
for i in range(n_gpus):
    name = torch.cuda.get_device_name(i)
    vram = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f"   GPU {i}: {name} | {vram:.1f} GB VRAM")

if device.type == 'cpu':
    print("\n⚠️  WARNING: No GPU detected!")
    print("   Go to Settings → Accelerator → GPU T4 x2 → Save")

torch.manual_seed(42)
np.random.seed(42)
print("\n✅ Imports complete!")



✅ Device: cuda
✅ GPUs available: 2
   GPU 0: Tesla T4 | 14.6 GB VRAM
   GPU 1: Tesla T4 | 14.6 GB VRAM

✅ Imports complete!


In [3]:

# ═════════════════════════════════════════════════════════════════════════════
# CELL 3 — CONFIGURATION + SAVE/LOAD HELPERS
# All file paths and model-saving functions defined here
# ═════════════════════════════════════════════════════════════════════════════

# ── All 12 LLM classes ──────────────────────────────────────────────────────
CLASSES = [
    'gpt2', 'llama-chat', 'human', 'chatgpt', 'mistral', 'gpt4',
    'mpt-chat', 'mistral-chat', 'gpt3', 'mpt', 'cohere-chat', 'cohere'
]
NUM_CLASSES = len(CLASSES)

# ── Directory structure ──────────────────────────────────────────────────────
WORKING    = "/kaggle/working"
MODELS_DIR = f"{WORKING}/models"      # All trained models saved here
CACHE_DIR  = f"{WORKING}/hf_cache"    # HuggingFace model cache
EMB_CACHE  = f"{WORKING}/emb_cache"   # Embedding numpy cache

for d in [MODELS_DIR, CACHE_DIR, EMB_CACHE]:
    os.makedirs(d, exist_ok=True)

print(f"📁 Models will be saved to: {MODELS_DIR}")
print(f"   (Download from Kaggle Output tab after training)")

# ── DeBERTa configs ──────────────────────────────────────────────────────────
DEBERTA_MINI_CONFIG = {
    "model_name":      "microsoft/deberta-v3-base",
    "max_length":      512,
    "batch_size":      16,    # per GPU; T4 16GB handles this
    "grad_accum":      2,     # effective batch = 32
    "num_epochs":      3,
    "learning_rate":   2e-5,
    "warmup_ratio":    0.1,
    "weight_decay":    0.01,
    "label_smoothing": 0.05,
}

DEBERTA_FULL_CONFIG = {
    "model_name":      "microsoft/deberta-v3-base",
    "max_length":      512,
    "batch_size":      8,
    "grad_accum":      8,     # effective batch = 64
    "num_epochs":      2,
    "learning_rate":   1e-5,
    "warmup_ratio":    0.06,
    "weight_decay":    0.01,
    "label_smoothing": 0.05,
}

# ── Global results tracker ───────────────────────────────────────────────────
# Stores: { "model_name": {"accuracy": 0.94, "stage": "mini", "saved_path": "..."} }
ALL_RESULTS = {}

def save_results():
    """Save all results to JSON after every model trains."""
    path = f"{MODELS_DIR}/results_all.json"
    with open(path, 'w') as f:
        json.dump(ALL_RESULTS, f, indent=2)
    print(f"   💾 Results saved → results_all.json")

def save_sklearn_model(model, filename, description=""):
    """Save any scikit-learn model (TF-IDF, LR, SVM, LightGBM)."""
    path = f"{MODELS_DIR}/{filename}"
    with open(path, 'wb') as f:
        pickle.dump(model, f)
    size_mb = os.path.getsize(path) / 1024**2
    print(f"   💾 Saved: {filename}  ({size_mb:.1f} MB)  ← {description}")
    return path

def load_sklearn_model(filename):
    """Load any saved scikit-learn model."""
    path = f"{MODELS_DIR}/{filename}"
    if not os.path.exists(path):
        raise FileNotFoundError(f"Model not found: {path}")
    with open(path, 'rb') as f:
        model = pickle.load(f)
    print(f"   ✅ Loaded: {filename}")
    return model

def save_pytorch_model(model, filename, description=""):
    """Save DeBERTa or any PyTorch model weights."""
    path = f"{MODELS_DIR}/{filename}"
    # Handle DataParallel wrapper
    raw = model.module if hasattr(model, 'module') else model
    torch.save(raw.state_dict(), path)
    size_mb = os.path.getsize(path) / 1024**2
    print(f"   💾 Saved: {filename}  ({size_mb:.1f} MB)  ← {description}")
    return path

def load_pytorch_model(filename, model_name, num_labels=12):
    """Load a saved DeBERTa model for inference."""
    path = f"{MODELS_DIR}/{filename}"
    if not os.path.exists(path):
        raise FileNotFoundError(f"Model not found: {path}")
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=num_labels,
        cache_dir=CACHE_DIR, ignore_mismatched_sizes=True
    )
    model.load_state_dict(torch.load(path, map_location=device))
    model = model.to(device)
    model.eval()
    print(f"   ✅ Loaded PyTorch model: {filename}")
    return model

def record_result(name, accuracy, stage, saved_path):
    """Track a result in the global results dict."""
    ALL_RESULTS[name] = {
        "accuracy":    round(float(accuracy), 6),
        "accuracy_pct": round(float(accuracy)*100, 2),
        "stage":       stage,
        "saved_path":  saved_path
    }
    print(f"   📊 Recorded: [{stage}] {name} → {accuracy*100:.2f}%")

print("✅ Configuration ready!")
print(f"   Model directory: {MODELS_DIR}")


📁 Models will be saved to: /kaggle/working/models
   (Download from Kaggle Output tab after training)
✅ Configuration ready!
   Model directory: /kaggle/working/models


In [5]:

df_full = pd.read_parquet('/kaggle/input/datasets/arjungangwar1/llm-generated-text/train.parquet')


print(f"\n📊 Full dataset: {len(df_full):,} rows")
print(f"   Classes present: {df_full['model'].nunique()}")
print(df_full['model'].value_counts().to_string())


df_val_full = pd.read_parquet('/kaggle/input/datasets/arjungangwar1/llm-generated-text/val.parquet')


print(f"\n📊 Full dataset: {len(df_val_full):,} rows")
print(f"   Classes present: {df_val_full['model'].nunique()}")
print(df_val_full['model'].value_counts().to_string())

df_test_full = pd.read_parquet('/kaggle/input/datasets/arjungangwar1/llm-generated-text/val.parquet')


print(f"\n📊 Full dataset: {len(df_test_full):,} rows")
print(f"   Classes present: {df_test_full['model'].nunique()}")
print(df_test_full['model'].value_counts().to_string())


📊 Full dataset: 796,800 rows
   Classes present: 12
model
mpt             66495
gpt3            66486
gpt2            66458
mistral-chat    66454
mistral         66439
chatgpt         66427
cohere          66409
human           66387
mpt-chat        66343
llama-chat      66323
cohere-chat     66290
gpt4            66289



📊 Full dataset: 99,600 rows
   Classes present: 12
model
llama-chat      8430
cohere          8425
cohere-chat     8339
mpt-chat        8335
mistral-chat    8319
gpt2            8294
mpt             8290
gpt4            8273
chatgpt         8256
human           8242
mistral         8212
gpt3            8185



📊 Full dataset: 99,600 rows
   Classes present: 12
model
llama-chat      8430
cohere          8425
cohere-chat     8339
mpt-chat        8335
mistral-chat    8319
gpt2            8294
mpt             8290
gpt4            8273
chatgpt         8256
human           8242
mistral         8212
gpt3            8185


In [7]:

# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — Load MINI DATASET ( fast testing)
# ─────────────────────────────────────────────────────────────────────────────

df_mini = pd.read_parquet('/kaggle/input/datasets/arjungangwar1/llm-generated-text/medium_train.parquet')
df_mini = df_mini.rename(columns={
    'text': 'generation',
    'generated_by': 'model'
})

print(df_mini.columns)
print(f"✅ Mini dataset: {len(df_mini):,} rows")
# print(f"   Distribution:\n{df_mini['model'].value_counts().to_string()}")
print("\nClass distribution:")
for cls, cnt in df_mini['model'].value_counts().sort_index().items():
    bar = '█' * (cnt // 50)
    print(f"  {cls:<15}: {cnt:4d}  {bar}")



df_mini_val = pd.read_parquet('/kaggle/input/datasets/arjungangwar1/llm-generated-text/medium_val.parquet')

df_mini_val = df_mini_val.rename(columns={
    'text': 'generation',
    'generated_by': 'model'
})
print(df_mini_val.columns)
print(f"✅ Mini dataset: {len(df_mini_val):,} rows")
# print(f"   Distribution:\n{df_mini_val['model'].value_counts().to_string()}")
print("\nClass distribution:")
for cls, cnt in df_mini_val['model'].value_counts().sort_index().items():
    bar = '█' * (cnt // 50)
    print(f"  {cls:<15}: {cnt:4d}  {bar}")





df_mini_test = pd.read_parquet('/kaggle/input/datasets/arjungangwar1/llm-generated-text/medium_test.parquet')

df_mini_test = df_mini_test.rename(columns={
    'text': 'generation',
    'generated_by': 'model'
})

print(df_mini_test.columns)
print(f"✅ Mini dataset: {len(df_mini_test):,} rows")
# print(f"   Distribution:\n{df_mini_test['model'].value_counts().to_string()}")
print("\nClass distribution:")
for cls, cnt in df_mini_test['model'].value_counts().sort_index().items():
    bar = '█' * (cnt // 50)
    print(f"  {cls:<15}: {cnt:4d}  {bar}")



# ── Label encoder (shared by ALL models throughout this notebook) ─────────────
le = LabelEncoder()
le.fit(CLASSES)

# Save label encoder — needed to decode predictions on test data
save_sklearn_model(le, "label_encoder.pkl",
                   "LabelEncoder — REQUIRED to decode any model's predictions")

X_mini_train = df_mini['generation'].values
y_mini_train = df_mini['model'].values
y_mini_train_enc = le.transform(y_mini_train)

X_mini_val = df_mini_val['generation'].values
y_mini_val = df_mini_val['model'].values
y_mini_val_enc = le.transform(y_mini_val)


print(f"\n✅ Mini split:")
print(f"   Train: {len(X_mini_train):,} samples")
print(f"   Val:   {len(X_mini_val):,} samples")
print(f"\n   Label mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")



Index(['generation', 'model'], dtype='object')
✅ Mini dataset: 79,681 rows

Class distribution:
  chatgpt        : 6643  ████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  cohere         : 6641  ████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  cohere-chat    : 6629  ████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  gpt2           : 6646  ████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  gpt3           : 6649  ████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  gpt4           : 6629  █████████████████████████████████████████████████████████████████████████████████████████

Index(['generation', 'model'], dtype='object')
✅ Mini dataset: 9,959 rows

Class distribution:
  chatgpt        :  826  ████████████████
  cohere         :  842  ████████████████
  cohere-chat    :  834  ████████████████
  gpt2           :  829  ████████████████
  gpt3           :  818  ████████████████
  gpt4           :  827  ████████████████
  human          :  824  ████████████████
  llama-chat     :  843  ████████████████
  mistral        :  821  ████████████████
  mistral-chat   :  832  ████████████████
  mpt            :  829  ████████████████
  mpt-chat       :  834  ████████████████
   💾 Saved: label_encoder.pkl  (0.0 MB)  ← LabelEncoder — REQUIRED to decode any model's predictions

✅ Mini split:
   Train: 79,681 samples
   Val:   9,959 samples

   Label mapping: {np.str_('chatgpt'): np.int64(0), np.str_('cohere'): np.int64(1), np.str_('cohere-chat'): np.int64(2), np.str_('gpt2'): np.int64(3), np.str_('gpt3'): np.int64(4), np.str_('gpt4'): np.int64(5), np.str_('human'): np.int

In [8]:
# ── Quick text stats ───────────────────────────────────────────────────────
df_mini['wc'] = df_mini['generation'].str.split().str.len()
print(f"\n📏 Text length stats:")
print(df_mini.groupby('model')['wc'].mean().round(1).to_string())


📏 Text length stats:
model
chatgpt         261.8
cohere          229.7
cohere-chat     186.6
gpt2            288.9
gpt3            133.8
gpt4            270.5
human           285.4
llama-chat      265.6
mistral         253.7
mistral-chat    200.7
mpt             276.3
mpt-chat        162.6


In [9]:
# ── Quick text stats ───────────────────────────────────────────────────────
df_mini_test['wc'] = df_mini_test['generation'].str.split().str.len()
print(f"\n📏 Text length stats:")
print(df_mini.groupby('model')['wc'].mean().round(1).to_string())


📏 Text length stats:
model
chatgpt         261.8
cohere          229.7
cohere-chat     186.6
gpt2            288.9
gpt3            133.8
gpt4            270.5
human           285.4
llama-chat      265.6
mistral         253.7
mistral-chat    200.7
mpt             276.3
mpt-chat        162.6


In [10]:
# ── Quick text stats ───────────────────────────────────────────────────────
df_mini_val['wc'] = df_mini_val['generation'].str.split().str.len()
print(f"\n📏 Text length stats:")
print(df_mini_val.groupby('model')['wc'].mean().round(1).to_string())


📏 Text length stats:
model
chatgpt         261.3
cohere          227.0
cohere-chat     183.5
gpt2            294.4
gpt3            129.0
gpt4            270.5
human           286.6
llama-chat      268.8
mistral         255.0
mistral-chat    198.7
mpt             267.2
mpt-chat        163.9


In [11]:


# ═════════════════════════════════════════════════════════════════════════════
# CELL 6 — TF-IDF BASELINE MODELS  (MINI DATASET)
# Models: NaiveBayes, LogisticRegression, LinearSVM
# ═════════════════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("📊  CELL 6: TF-IDF BASELINES  (Mini Dataset)")
print("="*60)
print("""
What is TF-IDF?
  → Converts text to numbers (word frequency matrix)
  → TF  = how often a word appears in THIS document
  → IDF = penalizes words that appear in ALL documents (they're not useful)
  → Result: high score for words that are DISTINCTIVE to a specific document
  
Why these 3 models?
  → NaiveBayes:  Fastest, probabilistic, good sanity check
  → LogReg:      Learns which words "vote" for each class
  → LinearSVM:   Best traditional classifier for text, maximum-margin boundary
""")

# ── Light text cleaning for TF-IDF ───────────────────────────────────────────
def clean_text(text):
    """Remove extra whitespace and lowercase. Keep punctuation (it's a signal!)"""
    return re.sub(r'\s+', ' ', str(text)).strip().lower()

X_mini_train_clean = [clean_text(t) for t in X_mini_train]
X_mini_val_clean   = [clean_text(t) for t in X_mini_val]

# ── TF-IDF Vectorizer (shared by all 3 models) ───────────────────────────────
print("⏳ Fitting TF-IDF vectorizer...")
tfidf = TfidfVectorizer(
    max_features  = 80000,   # Top 80k words/bigrams
    ngram_range   = (1, 2),  # Unigrams + bigrams ("I think" as one feature)
    sublinear_tf  = True,    # log(1+tf) — compresses large counts
    min_df        = 2,       # Ignore words in < 2 documents
    max_df        = 0.95,    # Ignore words in > 95% of documents
    strip_accents = 'unicode',
    token_pattern = r'\w{1,}',
)
X_tr_tfidf  = tfidf.fit_transform(X_mini_train_clean)
X_val_tfidf = tfidf.transform(X_mini_val_clean)

print(f"✅ TF-IDF matrix: {X_tr_tfidf.shape}")
print(f"   Vocabulary size: {len(tfidf.vocabulary_):,}")
print(f"   Matrix sparsity: {1 - X_tr_tfidf.nnz/(X_tr_tfidf.shape[0]*X_tr_tfidf.shape[1]):.2%}")

# Save TF-IDF vectorizer (needed to transform new/test data)
save_sklearn_model(tfidf, "tfidf_vectorizer.pkl",
                   "TF-IDF Vectorizer — use this to transform new test text")

# ── Model 1: Naive Bayes ──────────────────────────────────────────────────────
print("\n📌 Training: Naive Bayes...")
t0 = time.time()
nb_model = ComplementNB(alpha=0.1)
nb_model.fit(X_tr_tfidf, y_mini_train)
nb_preds = nb_model.predict(X_val_tfidf)
nb_acc   = accuracy_score(y_mini_val, nb_preds)
print(f"   ✅ Accuracy: {nb_acc:.4f} ({nb_acc*100:.2f}%)  | Time: {time.time()-t0:.1f}s")
print(f"   Per-class report:\n{classification_report(y_mini_val, nb_preds, digits=3)}")

path_nb = save_sklearn_model(nb_model, "mini_tfidf_nb.pkl",
                              f"Naive Bayes | acc={nb_acc:.4f}")
record_result("Mini: TF-IDF + NaiveBayes", nb_acc, "mini", path_nb)

# ── Model 2: Logistic Regression ──────────────────────────────────────────────
print("\n📌 Training: Logistic Regression...")
t0 = time.time()
lr_model = LogisticRegression(
    C=2.0, max_iter=1000, n_jobs=-1, random_state=42,
    multi_class='multinomial', solver='lbfgs'
)
lr_model.fit(X_tr_tfidf, y_mini_train)
lr_preds = lr_model.predict(X_val_tfidf)
lr_acc   = accuracy_score(y_mini_val, lr_preds)
print(f"   ✅ Accuracy: {lr_acc:.4f} ({lr_acc*100:.2f}%)  | Time: {time.time()-t0:.1f}s")
print(f"   Per-class report:\n{classification_report(y_mini_val, lr_preds, digits=3)}")

path_lr = save_sklearn_model(lr_model, "mini_tfidf_lr.pkl",
                              f"LogisticRegression | acc={lr_acc:.4f}")
record_result("Mini: TF-IDF + LogReg", lr_acc, "mini", path_lr)

# ── Model 3: Linear SVM ───────────────────────────────────────────────────────
print("\n📌 Training: Linear SVM...")
t0 = time.time()
svm_model = LinearSVC(C=1.0, max_iter=3000, random_state=42)
svm_model.fit(X_tr_tfidf, y_mini_train)
svm_preds = svm_model.predict(X_val_tfidf)
svm_acc   = accuracy_score(y_mini_val, svm_preds)
print(f"   ✅ Accuracy: {svm_acc:.4f} ({svm_acc*100:.2f}%)  | Time: {time.time()-t0:.1f}s")
print(f"   Per-class report:\n{classification_report(y_mini_val, svm_preds, digits=3)}")

path_svm = save_sklearn_model(svm_model, "mini_tfidf_svm.pkl",
                               f"LinearSVM | acc={svm_acc:.4f}")
record_result("Mini: TF-IDF + LinearSVM", svm_acc, "mini", path_svm)

save_results()

print(f"\n📊 TF-IDF Baseline Summary:")
print(f"   NaiveBayes:  {nb_acc*100:.2f}%")
print(f"   LogReg:      {lr_acc*100:.2f}%")
print(f"   LinearSVM:   {svm_acc*100:.2f}%  ← usually best")




📊  CELL 6: TF-IDF BASELINES  (Mini Dataset)

What is TF-IDF?
  → Converts text to numbers (word frequency matrix)
  → TF  = how often a word appears in THIS document
  → IDF = penalizes words that appear in ALL documents (they're not useful)
  → Result: high score for words that are DISTINCTIVE to a specific document
  
Why these 3 models?
  → NaiveBayes:  Fastest, probabilistic, good sanity check
  → LogReg:      Learns which words "vote" for each class
  → LinearSVM:   Best traditional classifier for text, maximum-margin boundary



⏳ Fitting TF-IDF vectorizer...


✅ TF-IDF matrix: (79681, 80000)
   Vocabulary size: 80,000
   Matrix sparsity: 99.71%
   💾 Saved: tfidf_vectorizer.pkl  (3.0 MB)  ← TF-IDF Vectorizer — use this to transform new test text

📌 Training: Naive Bayes...


   ✅ Accuracy: 0.4153 (41.53%)  | Time: 0.5s
   Per-class report:
              precision    recall  f1-score   support

     chatgpt      0.389     0.695     0.499       826
      cohere      0.444     0.280     0.344       842
 cohere-chat      0.588     0.173     0.267       834
        gpt2      0.329     0.519     0.403       829
        gpt3      0.280     0.507     0.361       818
        gpt4      0.566     0.579     0.573       827
       human      0.413     0.540     0.468       824
  llama-chat      0.478     0.495     0.486       843
     mistral      0.352     0.172     0.231       821
mistral-chat      0.546     0.242     0.335       832
         mpt      0.424     0.567     0.485       829
    mpt-chat      0.748     0.221     0.341       834

    accuracy                          0.415      9959
   macro avg      0.463     0.416     0.399      9959
weighted avg      0.464     0.415     0.399      9959

   💾 Saved: mini_tfidf_nb.pkl  (15.3 MB)  ← Naive Bayes | acc=0.415

   ✅ Accuracy: 0.6264 (62.64%)  | Time: 129.2s
   Per-class report:
              precision    recall  f1-score   support

     chatgpt      0.751     0.780     0.765       826
      cohere      0.515     0.513     0.514       842
 cohere-chat      0.528     0.459     0.491       834
        gpt2      0.574     0.577     0.575       829
        gpt3      0.572     0.697     0.628       818
        gpt4      0.780     0.817     0.798       827
       human      0.566     0.710     0.630       824
  llama-chat      0.753     0.749     0.751       843
     mistral      0.530     0.449     0.486       821
mistral-chat      0.595     0.570     0.582       832
         mpt      0.671     0.591     0.629       829
    mpt-chat      0.676     0.607     0.640       834

    accuracy                          0.626      9959
   macro avg      0.626     0.627     0.624      9959
weighted avg      0.626     0.626     0.624      9959

   💾 Saved: mini_tfidf_lr.pkl  (7.3 MB)  ← LogisticRegression | a

   ✅ Accuracy: 0.6270 (62.70%)  | Time: 25.0s
   Per-class report:
              precision    recall  f1-score   support

     chatgpt      0.749     0.794     0.771       826
      cohere      0.522     0.504     0.512       842
 cohere-chat      0.533     0.468     0.498       834
        gpt2      0.575     0.595     0.585       829
        gpt3      0.585     0.689     0.633       818
        gpt4      0.783     0.820     0.801       827
       human      0.572     0.676     0.620       824
  llama-chat      0.737     0.773     0.755       843
     mistral      0.521     0.429     0.470       821
mistral-chat      0.612     0.572     0.591       832
         mpt      0.622     0.598     0.610       829
    mpt-chat      0.683     0.607     0.643       834

    accuracy                          0.627      9959
   macro avg      0.624     0.627     0.624      9959
weighted avg      0.624     0.627     0.624      9959

   💾 Saved: mini_tfidf_svm.pkl  (7.3 MB)  ← LinearSVM | acc=0.6270

In [12]:

# ═════════════════════════════════════════════════════════════════════════════
# CELL 7 — SBERT EMBEDDINGS + STYLOMETRIC + LightGBM  (MINI DATASET)
# ═════════════════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("🧠  CELL 7: SBERT + STYLOMETRIC + LightGBM  (Mini Dataset)")
print("="*60)
print("""
What is SBERT?
  → Neural model that converts whole sentences to 384/768-dim vectors
  → Unlike TF-IDF, captures MEANING and STYLE (not just word counts)
  → "Certainly! I'd be happy to help" and "Sure, let me assist"
    map to SIMILAR vectors — something TF-IDF completely misses

What are stylometric features?
  → Hand-crafted text statistics: avg sentence length, vocab richness,
    punctuation rates, specific phrase patterns per LLM, etc.
  → Complements embeddings by capturing structural style

Why LightGBM?
  → Gradient boosting tree ensemble — handles non-linear patterns well
  → Works great on dense embedding vectors
  → Often beats neural networks on tabular/embedding features
""")

# ── Part A: Stylometric Feature Extraction ───────────────────────────────────
print("─"*50)
print("PART A: Extracting Stylometric Features")
print("─"*50)

# Phrases characteristic of specific LLMs
LLM_PHRASE_PATTERNS = {
    'certainly':    r'\bcertainly\b',
    'as_an_ai':     r'\bas an ai\b',
    'great_q':      r'\bgreat question\b',
    'id_be_happy':  r"i'?d be happy",
    'here_are':     r'\bhere are\b',
    'however':      r'\bhowever\b',
    'furthermore':  r'\bfurthermore\b|\bmoreover\b|\badditionally\b',
    'triple_repeat':r'(\b\w+\b) \1 \1',      # GPT-2 repetition bug
    'tbh_idk':      r'\btbh\b|\bidk\b|\bimo\b',
    'you_know':     r'\byou know\b|\bi mean\b',
    'primary_core': r'\bprimary\b|\bfundamental\b|\bcore\b',
    'methodology':  r'\bmethodology\b|\bframework\b',
    'thank_you':    r'\bthank you\b',
    'of_course':    r'\bof course\b',
    'let_me':       r'\blet me\b',
}

def extract_stylometric(text):
    """
    Extract 30 stylometric features from text.
    Returns a float32 numpy array of shape (30,)
    """
    if not isinstance(text, str) or len(text.strip()) == 0:
        return np.zeros(30, dtype=np.float32)

    t      = str(text).lower()
    words  = t.split()
    sents  = [s.strip() for s in re.split(r'[.!?]+', t) if len(s.strip()) > 4]
    nw     = max(len(words), 1)
    ns     = max(len(sents), 1)

    # Word stats
    word_lens = [len(w) for w in words if w.isalpha()]
    sent_lens = [len(s.split()) for s in sents]
    unique_words = len(set(w for w in words if w.isalpha()))

    # Base features (15 features)
    feats = [
        len(text),                                          # 1. char count
        nw,                                                 # 2. word count
        ns,                                                 # 3. sentence count
        np.mean(word_lens) if word_lens else 0,             # 4. avg word length
        unique_words / nw,                                  # 5. vocabulary richness (TTR)
        np.mean(sent_lens) if sent_lens else 0,             # 6. avg sentence length
        np.std(sent_lens) if len(sent_lens) > 1 else 0,    # 7. sentence length variance
        text.count(',')  / nw,                             # 8. comma rate
        text.count('!')  / ns,                             # 9. exclamation rate
        text.count('?')  / ns,                             # 10. question rate
        text.count(':')  / ns,                             # 11. colon rate
        len(re.findall(r'^\s*[-•*]\s', text, re.M)) / ns, # 12. bullet point rate
        len(re.findall(r'^\s*\d+[.)]\s', text, re.M)) / ns, # 13. numbered list rate
        sum(1 for w in words if w.isupper() and len(w)>1) / nw,  # 14. ALL CAPS rate
        text.count('...') / ns,                            # 15. ellipsis rate
    ]

    # LLM-specific phrase features (15 features, one per phrase pattern)
    for pattern in LLM_PHRASE_PATTERNS.values():
        count = len(re.findall(pattern, t))
        feats.append(min(count / ns, 5.0))   # normalize by sentence count, cap at 5

    return np.array(feats, dtype=np.float32)

# Extract for mini dataset
print("⏳ Extracting stylometric features (fast, ~10s)...")
t0 = time.time()
X_mini_tr_stylo  = np.vstack([extract_stylometric(t) for t in X_mini_train])
X_mini_val_stylo = np.vstack([extract_stylometric(t) for t in X_mini_val])

# Normalize (StandardScaler: mean=0, std=1 for each feature)
stylo_scaler = StandardScaler()
X_mini_tr_stylo_n  = stylo_scaler.fit_transform(X_mini_tr_stylo)
X_mini_val_stylo_n = stylo_scaler.transform(X_mini_val_stylo)

# Save scaler (needed to transform test data identically)
save_sklearn_model(stylo_scaler, "stylo_scaler.pkl",
                   "StandardScaler for stylometric features — use when predicting")

print(f"✅ Stylometric done in {time.time()-t0:.1f}s")
print(f"   Feature shape: {X_mini_tr_stylo.shape}  (30 features per text)")

# ── Part B: SBERT Embedding Extraction (Multi-GPU) ───────────────────────────
print("\n─"*25)
print("PART B: SBERT Embeddings  (Multi-GPU)")
print("─"*50)

SBERT_MODELS = {
    "MiniLM":    "all-MiniLM-L6-v2",      # 384d — fastest (~3min T4)
    "MPNet":     "all-mpnet-base-v2",      # 768d — strong  (~8min T4)
    "BGE-base":  "BAAI/bge-base-en-v1.5", # 768d — best open-source (~8min T4)
}

def encode_with_sbert(train_texts, val_texts, model_name, tag,
                      batch_size=256, max_chars=2000):
    """
    Encode texts using SBERT.
    Uses multi-GPU pool if 2+ GPUs available (2x speedup).
    Caches results to disk to avoid re-encoding.
    Returns: train_emb (float32), val_emb (float32)
    """
    cache_tr  = f"{EMB_CACHE}/{tag}_train.npy"
    cache_val = f"{EMB_CACHE}/{tag}_val.npy"

    # ── Try cache first ───────────────────────────────────────────────────
    if os.path.exists(cache_tr) and os.path.exists(cache_val):
        print(f"   📦 Cache hit → loading {tag} embeddings from disk")
        return (np.load(cache_tr).astype(np.float32),
                np.load(cache_val).astype(np.float32))

    print(f"   ⏳ Loading {model_name}...")
    sbert = SentenceTransformer(model_name)

    tr_trunc  = [str(t)[:max_chars] for t in train_texts]
    val_trunc = [str(t)[:max_chars] for t in val_texts]

    if n_gpus > 1:
        # ── MULTI-GPU mode (both T4s work in parallel) ────────────────────
        print(f"   🚀 Multi-GPU encoding on {n_gpus} GPUs!")
        gpu_ids = [f'cuda:{i}' for i in range(n_gpus)]
        pool = sbert.start_multi_process_pool(target_devices=gpu_ids)

        t0 = time.time()
        tr_emb = sbert.encode_multi_process(
            tr_trunc, pool=pool, batch_size=batch_size, normalize_embeddings=True)
        print(f"   Train: {len(tr_trunc):,} texts → {time.time()-t0:.1f}s")

        t0 = time.time()
        val_emb = sbert.encode_multi_process(
            val_trunc, pool=pool, batch_size=batch_size, normalize_embeddings=True)
        print(f"   Val:   {len(val_trunc):,} texts → {time.time()-t0:.1f}s")

        sbert.stop_multi_process_pool(pool)    # always stop pool!

    else:
        # ── Single GPU fallback ───────────────────────────────────────────
        print(f"   ⚠️  Single GPU mode")
        t0 = time.time()
        tr_emb  = sbert.encode(tr_trunc,  batch_size=batch_size,
                               show_progress_bar=True, normalize_embeddings=True,
                               device='cuda' if torch.cuda.is_available() else 'cpu')
        val_emb = sbert.encode(val_trunc, batch_size=batch_size,
                               show_progress_bar=True, normalize_embeddings=True,
                               device='cuda' if torch.cuda.is_available() else 'cpu')
        print(f"   Done in {time.time()-t0:.1f}s")

    tr_emb  = np.array(tr_emb, dtype=np.float32)
    val_emb = np.array(val_emb, dtype=np.float32)

    np.save(cache_tr, tr_emb)
    np.save(cache_val, val_emb)
    print(f"   ✅ Shape: {tr_emb.shape}  | Cached to disk")

    del sbert; gc.collect(); torch.cuda.empty_cache()
    return tr_emb, val_emb

# ── Train LR on each embedding model + stylometric ───────────────────────────
print("\n⏳ Testing 3 SBERT embedding models...")

emb_results_mini = {}
best_emb_name    = None
best_emb_acc     = 0.0
best_tr_comb     = None
best_val_comb    = None

for tag, model_name in SBERT_MODELS.items():
    print(f"\n{'─'*50}")
    print(f"  Testing: {tag}  →  {model_name}")
    print(f"{'─'*50}")

    try:
        tr_emb, val_emb = encode_with_sbert(
            X_mini_train, X_mini_val, model_name, f"mini_{tag}", batch_size=256)

        # Combine embedding + stylometric
        tr_comb  = np.hstack([tr_emb,  X_mini_tr_stylo_n])
        val_comb = np.hstack([val_emb, X_mini_val_stylo_n])
        print(f"  Combined features: {tr_comb.shape}")

        # Quick Logistic Regression
        lr_e = LogisticRegression(C=5.0, max_iter=1000, n_jobs=-1, random_state=42)
        lr_e.fit(tr_comb, y_mini_train)
        acc = accuracy_score(y_mini_val, lr_e.predict(val_comb))

        # Save this LR model
        path_lr_e = save_sklearn_model(
            lr_e, f"mini_lr_{tag.lower()}.pkl",
            f"LR on {tag}+Stylo | acc={acc:.4f}")
        record_result(f"Mini: LR + {tag} + Stylo", acc, "mini", path_lr_e)

        emb_results_mini[tag] = acc
        print(f"  ✅ LR accuracy: {acc:.4f} ({acc*100:.2f}%)")

        if acc > best_emb_acc:
            best_emb_acc  = acc
            best_emb_name = tag
            best_tr_comb  = tr_comb
            best_val_comb = val_comb

    except Exception as e:
        print(f"  ⚠️  {tag} failed: {e}")
        import traceback; traceback.print_exc()

print(f"\n🏆 Best SBERT model: {best_emb_name} ({best_emb_acc:.4f})")

# ── Part C: LightGBM on best embedding ───────────────────────────────────────
print("\n─"*25)
print(f"PART C: LightGBM on best embedding ({best_emb_name})")
print("─"*50)

lgb_mini_acc = 0.0
lgb_mini_model = None

if best_tr_comb is not None:
    lgb_mini_model = lgb.LGBMClassifier(
        n_estimators   = 800,
        learning_rate  = 0.04,
        num_leaves     = 127,
        max_depth      = -1,
        colsample_bytree = 0.8,
        subsample      = 0.8,
        min_child_samples = 20,
        reg_alpha      = 0.1,
        reg_lambda     = 0.1,
        random_state   = 42,
        n_jobs         = -1,
        verbose        = -1,
    )

    t0 = time.time()
    try:
        lgb_mini_model.fit(
            best_tr_comb, y_mini_train_enc,
            eval_set=[(best_val_comb, y_mini_val_enc)],
            callbacks=[lgb.early_stopping(40, verbose=False),
                       lgb.log_evaluation(100)]
        )
    except TypeError:
        lgb_mini_model.fit(
            best_tr_comb, y_mini_train_enc,
            eval_set=[(best_val_comb, y_mini_val_enc)],
            early_stopping_rounds=40, verbose=100
        )

    lgb_preds     = le.inverse_transform(lgb_mini_model.predict(best_val_comb))
    lgb_mini_acc  = accuracy_score(y_mini_val, lgb_preds)
    elapsed       = time.time() - t0

    print(f"✅ LightGBM accuracy: {lgb_mini_acc:.4f} ({lgb_mini_acc*100:.2f}%)")
    print(f"   Training time: {elapsed:.1f}s  | Best iteration: {lgb_mini_model.best_iteration_}")
    print(f"\n{classification_report(y_mini_val, lgb_preds, digits=3)}")

    path_lgb = save_sklearn_model(lgb_mini_model, "mini_lgbm_best.pkl",
                                  f"LightGBM on {best_emb_name}+Stylo | acc={lgb_mini_acc:.4f}")
    record_result(f"Mini: LightGBM + {best_emb_name} + Stylo",
                  lgb_mini_acc, "mini", path_lgb)
    save_results()



🧠  CELL 7: SBERT + STYLOMETRIC + LightGBM  (Mini Dataset)

What is SBERT?
  → Neural model that converts whole sentences to 384/768-dim vectors
  → Unlike TF-IDF, captures MEANING and STYLE (not just word counts)
  → "Certainly! I'd be happy to help" and "Sure, let me assist"
    map to SIMILAR vectors — something TF-IDF completely misses

What are stylometric features?
  → Hand-crafted text statistics: avg sentence length, vocab richness,
    punctuation rates, specific phrase patterns per LLM, etc.
  → Complements embeddings by capturing structural style

Why LightGBM?
  → Gradient boosting tree ensemble — handles non-linear patterns well
  → Works great on dense embedding vectors
  → Often beats neural networks on tabular/embedding features

──────────────────────────────────────────────────
PART A: Extracting Stylometric Features
──────────────────────────────────────────────────
⏳ Extracting stylometric features (fast, ~10s)...


   💾 Saved: stylo_scaler.pkl  (0.0 MB)  ← StandardScaler for stylometric features — use when predicting
✅ Stylometric done in 60.4s
   Feature shape: (79681, 30)  (30 features per text)

─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
PART B: SBERT Embeddings  (Multi-GPU)
──────────────────────────────────────────────────

⏳ Testing 3 SBERT embedding models...

──────────────────────────────────────────────────
  Testing: MiniLM  →  all-MiniLM-L6-v2
──────────────────────────────────────────────────
   ⏳ Loading all-MiniLM-L6-v2...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

[300]	valid_0's multi_logloss: 1.14008


[400]	valid_0's multi_logloss: 1.11617


[500]	valid_0's multi_logloss: 1.10985


✅ LightGBM accuracy: 0.6088 (60.88%)
   Training time: 2706.7s  | Best iteration: 519

              precision    recall  f1-score   support

     chatgpt      0.666     0.607     0.635       826
      cohere      0.484     0.458     0.471       842
 cohere-chat      0.435     0.456     0.445       834
        gpt2      0.653     0.630     0.641       829
        gpt3      0.577     0.711     0.637       818
        gpt4      0.662     0.653     0.657       827
       human      0.685     0.778     0.728       824
  llama-chat      0.652     0.737     0.692       843
     mistral      0.596     0.565     0.580       821
mistral-chat      0.513     0.490     0.501       832
         mpt      0.759     0.649     0.700       829
    mpt-chat      0.650     0.576     0.610       834

    accuracy                          0.609      9959
   macro avg      0.611     0.609     0.608      9959
weighted avg      0.611     0.609     0.608      9959



   💾 Saved: mini_lgbm_best.pkl  (84.0 MB)  ← LightGBM on BGE-base+Stylo | acc=0.6088
   📊 Recorded: [mini] Mini: LightGBM + BGE-base + Stylo → 60.88%
   💾 Results saved → results_all.json


In [14]:

# ═════════════════════════════════════════════════════════════════════════════
# CELL 9 — MINI ENSEMBLE + RESULTS SUMMARY
# ═════════════════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("🎯  CELL 9: MINI ENSEMBLE + RESULTS SUMMARY")
print("="*60)
print("""
What is a Soft-Voting Ensemble?
  → Each model outputs a PROBABILITY DISTRIBUTION over 12 classes
  → We take a WEIGHTED AVERAGE of those probabilities
  → Final prediction = class with highest average probability
  → Better than any single model because different models make
    different errors, and averaging CANCELS OUT individual mistakes
    
Weights: DeBERTa gets highest weight (most accurate),
         LightGBM second, TF-IDF SVM third.
""")

def softmax_transform(x):
    """Convert SVM decision function scores to pseudo-probabilities."""
    e = np.exp(x - x.max(1, keepdims=True))
    return (e / e.sum(1, keepdims=True)).astype(np.float32)

# ── Get DeBERTa probabilities ─────────────────────────────────────────────────
print("⏳ Loading mini DeBERTa for ensemble...")
deberta_ens = AutoModelForSequenceClassification.from_pretrained(
    DEBERTA_MINI_CONFIG['model_name'],
    num_labels=NUM_CLASSES, cache_dir=CACHE_DIR, ignore_mismatched_sizes=True
)
deberta_ens.load_state_dict(
    torch.load(f"{MODELS_DIR}/mini_deberta_best.pt", map_location=device))
deberta_ens = deberta_ens.to(device)
deberta_ens.eval()

val_dl_ens = DataLoader(
    LLMDataset(X_mini_val, y_mini_val_enc, tok_mini, DEBERTA_MINI_CONFIG['max_length']),
    batch_size=32, shuffle=False, num_workers=2
)

deberta_probs = []
with torch.no_grad():
    for batch in val_dl_ens:
        out   = deberta_ens(input_ids=batch['input_ids'].to(device),
                             attention_mask=batch['attention_mask'].to(device))
        probs = torch.softmax(out.logits, 1).cpu().numpy()
        deberta_probs.append(probs)
deberta_probs = np.vstack(deberta_probs)
print(f"✅ DeBERTa probs: {deberta_probs.shape}")

del deberta_ens; gc.collect(); torch.cuda.empty_cache()

# ── LightGBM probabilities ────────────────────────────────────────────────────
lgb_probs = None
if lgb_mini_model is not None and best_val_comb is not None:
    lgb_probs = lgb_mini_model.predict_proba(best_val_comb).astype(np.float32)
    print(f"✅ LightGBM probs: {lgb_probs.shape}")

# ── SVM pseudo-probabilities ──────────────────────────────────────────────────
svm_df    = svm_model.decision_function(X_val_tfidf)
svm_probs = softmax_transform(svm_df)
print(f"✅ SVM probs: {svm_probs.shape}")

# ── Weighted soft voting ──────────────────────────────────────────────────────
print("\nBuilding ensemble (weighted soft voting)...")

if lgb_probs is not None:
    # DeBERTa 65% + LightGBM 25% + SVM 10%
    ens_probs = (deberta_probs * 0.65 +
                 lgb_probs    * 0.25 +
                 svm_probs    * 0.10)
    ens_label = "DeBERTa(65%) + LGBM(25%) + SVM(10%)"
else:
    # DeBERTa 80% + SVM 20%
    ens_probs = deberta_probs * 0.80 + svm_probs * 0.20
    ens_label = "DeBERTa(80%) + SVM(20%)"

ens_preds = le.inverse_transform(ens_probs.argmax(1))
ens_acc   = accuracy_score(y_mini_val, ens_preds)

print(f"\n🏆 Ensemble accuracy: {ens_acc:.4f} ({ens_acc*100:.2f}%)")
print(f"   Config: {ens_label}")
print(f"\n{classification_report(y_mini_val, ens_preds, digits=3)}")

record_result(f"Mini: Ensemble ({ens_label})", ens_acc, "mini", "N/A — computed at runtime")
save_results()

# ── Mini results summary table ────────────────────────────────────────────────
print("\n" + "═"*65)
print("📊  MINI DATASET — ALL RESULTS SUMMARY")
print("═"*65)

mini_results = {k: v for k, v in ALL_RESULTS.items() if v['stage'] == 'mini'}
sorted_results = sorted(mini_results.items(), key=lambda x: x[1]['accuracy'], reverse=True)

print(f"\n{'Model':<50} {'Accuracy':>10}  Bar")
print("─"*75)
for name, info in sorted_results:
    acc  = info['accuracy']
    bar  = '█' * int(acc*30) + '░' * (30-int(acc*30))
    print(f"{name:<50} {acc*100:>9.2f}%  {bar}")

best_mini = sorted_results[0]
print(f"\n🏆 Best mini model: {best_mini[0]}")
print(f"   Accuracy: {best_mini[1]['accuracy']*100:.2f}%")
print(f"   Saved at: {best_mini[1]['saved_path']}")

if mini_deberta_acc < 0.70:
    print("\n⚠️  Mini DeBERTa accuracy < 70% — check your data before running full training!")
else:
    print(f"\n✅ Mini pipeline working well ({mini_deberta_acc*100:.1f}%)")
    print("   ▶  Ready for STAGE B: Full Dataset Training (Cells 10-13)")



🎯  CELL 9: MINI ENSEMBLE + RESULTS SUMMARY

What is a Soft-Voting Ensemble?
  → Each model outputs a PROBABILITY DISTRIBUTION over 12 classes
  → We take a WEIGHTED AVERAGE of those probabilities
  → Final prediction = class with highest average probability
  → Better than any single model because different models make
    different errors, and averaging CANCELS OUT individual mistakes
    
Weights: DeBERTa gets highest weight (most accurate),
         LightGBM second, TF-IDF SVM third.

⏳ Loading mini DeBERTa for ensemble...


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias        

✅ DeBERTa probs: (9959, 12)


✅ LightGBM probs: (9959, 12)
✅ SVM probs: (9959, 12)

Building ensemble (weighted soft voting)...

🏆 Ensemble accuracy: 0.0829 (8.29%)
   Config: DeBERTa(65%) + LGBM(25%) + SVM(10%)

              precision    recall  f1-score   support

     chatgpt      0.083     1.000     0.153       826
      cohere      0.000     0.000     0.000       842
 cohere-chat      0.000     0.000     0.000       834
        gpt2      0.000     0.000     0.000       829
        gpt3      0.000     0.000     0.000       818
        gpt4      0.000     0.000     0.000       827
       human      0.000     0.000     0.000       824
  llama-chat      0.000     0.000     0.000       843
     mistral      0.000     0.000     0.000       821
mistral-chat      0.000     0.000     0.000       832
         mpt      0.000     0.000     0.000       829
    mpt-chat      0.000     0.000     0.000       834

    accuracy                          0.083      9959
   macro avg      0.007     0.083     0.013      9959
weigh

In [15]:

# ═════════════════════════════════════════════════════════════════════════════
# ███████████████████████████  STAGE B  ████████████████████████████████████
# FULL DATASET TRAINING  (960,000 rows)
# Only run this after mini pipeline confirms everything works!
# ═════════════════════════════════════════════════════════════════════════════

print("\n\n" + "█"*65)
print("█" + " "*20 + "STAGE B: FULL DATASET" + " "*22 + "█")
print("█"*65)
print("""
⚠️  IMPORTANT — Read before running:
  1. Make sure mini pipeline completed successfully (cells 6-9)
  2. Full DeBERTa training takes 3-5 hours on T4 GPU
  3. Kaggle free session = 9 hours — plan accordingly
  4. All models saved with checkpoints → can resume if session ends
  5. Full dataset = ~960,000 rows (vs 12,000 in mini)
""")





█████████████████████████████████████████████████████████████████
█                    STAGE B: FULL DATASET                      █
█████████████████████████████████████████████████████████████████

⚠️  IMPORTANT — Read before running:
  1. Make sure mini pipeline completed successfully (cells 6-9)
  2. Full DeBERTa training takes 3-5 hours on T4 GPU
  3. Kaggle free session = 9 hours — plan accordingly
  4. All models saved with checkpoints → can resume if session ends
  5. Full dataset = ~960,000 rows (vs 12,000 in mini)



In [16]:


# ═════════════════════════════════════════════════════════════════════════════
# CELL 10 — FULL TF-IDF MODELS
# ═════════════════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("📊  CELL 10: FULL DATASET TF-IDF MODELS")
print("="*60)

# Prepare full dataset split
print(f"📂 Full dataset: {len(df_full):,} rows")

X_full_text = df_full['generation'].values
y_full_str  = df_full['model'].values
y_full_enc  = le.transform(y_full_str)

(X_full_train, X_full_val,
 y_full_train, y_full_val,
 y_full_train_enc, y_full_val_enc) = train_test_split(
    X_full_text, y_full_str, y_full_enc,
    test_size=0.10, random_state=42, stratify=y_full_enc
)

print(f"✅ Full split:")
print(f"   Train: {len(X_full_train):,}")
print(f"   Val:   {len(X_full_val):,}")

# Preprocess full text
print("\n⏳ Preprocessing full text (lowercasing + whitespace)...")
t0 = time.time()
X_full_train_clean = [clean_text(t) for t in X_full_train]
X_full_val_clean   = [clean_text(t) for t in X_full_val]
print(f"   Done in {time.time()-t0:.1f}s")

# Build full TF-IDF
print("\n⏳ Fitting TF-IDF on full training data...")
t0 = time.time()
tfidf_full = TfidfVectorizer(
    max_features=100000, ngram_range=(1,2),
    sublinear_tf=True, min_df=3, max_df=0.95,
    strip_accents='unicode', token_pattern=r'\w{1,}'
)
X_full_tr_tfidf  = tfidf_full.fit_transform(X_full_train_clean)
X_full_val_tfidf = tfidf_full.transform(X_full_val_clean)
print(f"✅ Full TF-IDF matrix: {X_full_tr_tfidf.shape}  ({time.time()-t0:.1f}s)")

# Save full TF-IDF vectorizer
save_sklearn_model(tfidf_full, "full_tfidf_vectorizer.pkl",
                   "Full TF-IDF Vectorizer (trained on 960k rows)")

# Full LinearSVM (usually the best TF-IDF model)
print("\n📌 Training Full LinearSVM...")
t0 = time.time()
svm_full = LinearSVC(C=1.0, max_iter=3000, random_state=42)
svm_full.fit(X_full_tr_tfidf, y_full_train)
svm_full_preds = svm_full.predict(X_full_val_tfidf)
svm_full_acc   = accuracy_score(y_full_val, svm_full_preds)
print(f"✅ Full LinearSVM accuracy: {svm_full_acc:.4f} ({svm_full_acc*100:.2f}%)  [{time.time()-t0:.1f}s]")
print(f"\n{classification_report(y_full_val, svm_full_preds, digits=3)}")

path_svm_full = save_sklearn_model(svm_full, "full_tfidf_svm.pkl",
                                    f"Full LinearSVM | acc={svm_full_acc:.4f}")
record_result("Full: TF-IDF + LinearSVM", svm_full_acc, "full", path_svm_full)
save_results()




📊  CELL 10: FULL DATASET TF-IDF MODELS
📂 Full dataset: 796,800 rows


✅ Full split:
   Train: 717,120
   Val:   79,680

⏳ Preprocessing full text (lowercasing + whitespace)...


   Done in 71.3s

⏳ Fitting TF-IDF on full training data...


✅ Full TF-IDF matrix: (717120, 100000)  (446.7s)


   💾 Saved: full_tfidf_vectorizer.pkl  (3.8 MB)  ← Full TF-IDF Vectorizer (trained on 960k rows)

📌 Training Full LinearSVM...


✅ Full LinearSVM accuracy: 0.7871 (78.71%)  [1692.1s]



              precision    recall  f1-score   support

     chatgpt      0.862     0.890     0.876      6643
      cohere      0.759     0.743     0.751      6641
 cohere-chat      0.761     0.716     0.738      6629
        gpt2      0.734     0.762     0.748      6646
        gpt3      0.776     0.851     0.812      6649
        gpt4      0.878     0.911     0.894      6629
       human      0.780     0.867     0.821      6639
  llama-chat      0.851     0.870     0.861      6632
     mistral      0.701     0.600     0.647      6644
mistral-chat      0.789     0.750     0.769      6645
         mpt      0.742     0.729     0.736      6649
    mpt-chat      0.793     0.756     0.774      6634

    accuracy                          0.787     79680
   macro avg      0.786     0.787     0.785     79680
weighted avg      0.786     0.787     0.785     79680

   💾 Saved: full_tfidf_svm.pkl  (9.2 MB)  ← Full LinearSVM | acc=0.7871
   📊 Recorded: [full] Full: TF-IDF + LinearSVM → 78.71%
   💾